# Install dependencies

In [3]:
!pip install tensorflow

In [4]:
!pip install numpy

In [5]:
!pip install pillow

# Resize Images

In [ ]:
import os
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

def resize_images_to_jpg(input_folder, output_folder, new_size=(500, 700)):
    """
    Resize all image files in the input folder to JPG format and save them to the output folder,
    maintaining the original folder structure.

    Args:
        input_folder (str): Path to the folder containing input images.
        output_folder (str): Path to the root folder to save resized JPG images.
        new_size (tuple): Target size for resizing (width, height).
    """
    # Get a list of all image files in the input folder (recursively)
    image_files = []
    for root, _, files in os.walk(input_folder):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.webp')):
                image_files.append(os.path.join(root, file))

    skipped_files = []  # To store files that are skipped
    to_process_files = []  # Files that need processing

    # Pre-process the files ahead of time
    for input_path in image_files:
        # Calculate the relative path from the input folder
        relative_path = os.path.relpath(os.path.dirname(input_path), input_folder)

        # Create the corresponding output directory
        output_dir = os.path.join(output_folder, relative_path)
        os.makedirs(output_dir, exist_ok=True)

        # Determine the output file path
        output_file_name = os.path.splitext(os.path.basename(input_path))[0] + ".jpg"
        output_path = os.path.join(output_dir, output_file_name)

        # Add to the process list if file doesn't exist, else add to skipped list
        if os.path.exists(output_path):
            skipped_files.append(input_path)
        else:
            to_process_files.append(input_path)

    # Update the progress bar dynamically to reflect the total number of images
    with tqdm(total=len(image_files), desc="Processing images", unit="file", leave=True, position=0, ncols=100) as pbar:
        for input_path in image_files:
            if input_path in skipped_files:
                # If the file was skipped, update progress bar but do not process
                pbar.update(1)
                continue

            try:
                # Open the image
                with Image.open(input_path) as img:
                    # Resize the image using LANCZOS resampling
                    img_resized = img.resize(new_size, Image.Resampling.LANCZOS)

                    # Convert to RGB if not already in RGB mode
                    if img.mode != 'RGB':
                        img_resized = img_resized.convert('RGB')

                    # Determine the relative path for saving
                    relative_path = os.path.relpath(os.path.dirname(input_path), input_folder)
                    output_dir = os.path.join(output_folder, relative_path)
                    os.makedirs(output_dir, exist_ok=True)
                    output_file_name = os.path.splitext(os.path.basename(input_path))[0] + ".jpg"
                    output_path = os.path.join(output_dir, output_file_name)

                    # Save as JPG
                    img_resized.save(output_path, format="JPEG")

            except UnidentifiedImageError:
                pass  # Skipping invalid image files silently
            except Exception as e:
                tqdm.write(f"Error processing {input_path}: {e}")
            finally:
                pbar.update(1)

    # If there were skipped files, you can print them at the end or handle them as needed
    if skipped_files:
        tqdm.write(f"Skipped {len(skipped_files)} files as they already exist.")

if __name__ == "__main__":
    input_folder = "/content/drive/My Drive/Card Database"  # Replace with your input folder path
    output_folder = "/content/drive/My Drive/data"  # Replace with your output folder path

    # Prompt the user for custom size input
    width = int(input("Enter the desired width (default 500): ") or 500)
    height = int(input("Enter the desired height (default 700): ") or 700)

    resize_images_to_jpg(input_folder, output_folder, new_size=(width, height))

Enter the desired width (default 500): 200
Enter the desired height (default 700): 200


Processing images:  93%|███████████████████████████████▋  | 92747/99435 [5:42:03<34:31,  3.23file/s]

# Run to Train model

In [8]:
import os
import random
import numpy as np
from PIL import Image
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import json
import gc
from tensorflow.keras import backend as K
import tensorflow as tf

# Path for saving JSON to Google Drive
GOOGLE_DRIVE_PATH = 'G:/My Drive/model/google_values.json'
MODELS_DIR = "G:/My Drive/model"
DATASET_PATH = "G:/My Drive/data"

# JSON Helpers
def load_values_from_json(file_path, default_values):
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r') as file:
                return json.load(file)
        except (json.JSONDecodeError, IOError) as e:
            print(f"Error loading JSON from {file_path}: {e}")
    return default_values

def save_values_to_json(file_path, values):
    try:
        with open(file_path, 'w') as file:
            json.dump(values, file, indent=4)
    except IOError as e:
        print(f"Error saving JSON to {file_path}: {e}")

# Training class
class TrainingThread:
    def __init__(self, model, data_path, target_size, epochs, class_labels, max_steps_per_epoch, json_file):
        self.model = model
        self.data_path = data_path
        self.target_size = target_size
        self.epochs = epochs
        self.class_labels = class_labels
        self.max_steps_per_epoch = max_steps_per_epoch
        self.json_file = json_file
        default_values = {"genome": 1, "full_iteration": 0}
        values = load_values_from_json(json_file, default_values)
        self.genome = values["genome"]
        self.full_iteration = values["full_iteration"]

    def create_data_generator(self):
        """Create a data generator for loading and augmenting images on the fly."""
        datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)  # Rescale and split for validation

        train_generator = datagen.flow_from_directory(
            self.data_path,
            target_size=self.target_size,
            batch_size=32,  # Adjust batch size as needed
            class_mode='categorical',
            subset='training',  # Use training subset
            shuffle=True
        )

        return train_generator

    def train_model(self):
        try:
            if not os.path.exists(MODELS_DIR):
                os.makedirs(MODELS_DIR)

            total_steps = self.max_steps_per_epoch * self.epochs
            current_step = 0
            train_generator = self.create_data_generator()  # Use data generator for loading batches

            for epoch in range(self.epochs):
                steps_in_epoch = min(self.max_steps_per_epoch, len(train_generator))

                for step in range(steps_in_epoch):
                    img_batch, label_batch = next(train_generator)  # Get next batch

                    loss, accuracy = self.model.train_on_batch(img_batch, label_batch)

                    print(f"Epoch {epoch + 1}/{self.epochs}, Step {current_step + 1}/{total_steps}: Loss={loss:.4f}, Accuracy={accuracy:.4f}")
                    current_step += 1

                    progress_percent = int((current_step / total_steps) * 100)
                    print(f"Progress: {progress_percent}%")

                    self.full_iteration += 1
                    save_values_to_json(self.json_file, {"genome": self.genome, "full_iteration": self.full_iteration})

                model_filename = f'card_predictor_model_genome_{self.genome}_epoch_{epoch + 1}_full_iteration_{self.full_iteration}.keras'
                model_path = os.path.join(MODELS_DIR, model_filename)
                self.model.save(model_path)
                print(f"Model saved at: {model_path}")

                self.genome += 1
                save_values_to_json(self.json_file, {"genome": self.genome, "full_iteration": self.full_iteration})
                print(f"Epoch {epoch + 1} completed. Genome updated to: {self.genome}, Full Iteration updated to: {self.full_iteration}")

                # Clear Keras session and perform garbage collection after each epoch
                K.clear_session()  # Clear Keras session to release resources
                gc.collect()  # Collect garbage to free unused memory

                # Reset the TensorFlow session explicitly if needed
                tf.keras.backend.clear_session()

            print("Training complete.")
        except Exception as e:
            print(f"Error during training: {str(e)}")

# Application for training (no GUI)
class CardPredictorApp:
    def __init__(self):
        self.dataset_path = DATASET_PATH
        self.target_size = (200, 200)
        self.class_labels = self.get_class_labels()
        self.model = self.load_or_create_model()

        default_values = {"genome": 1, "full_iteration": 0}
        values = load_values_from_json(GOOGLE_DRIVE_PATH, default_values)
        self.genome = values["genome"]
        self.full_iteration = values["full_iteration"]
        self.update_genome(self.genome)
        self.update_full_iteration(self.full_iteration)

    def get_class_labels(self):
        labels = []
        for subdir in os.listdir(self.dataset_path):
            if os.path.isdir(os.path.join(self.dataset_path, subdir)):
                labels.append(subdir)
        return sorted(labels)

    def load_or_create_model(self):
        model_file = os.path.join(MODELS_DIR, 'card_predictor_model.keras')
        if os.path.exists(model_file):
            return load_model(model_file)
        else:
            return self.build_model()

    def build_model(self):
        model = Sequential([
            Input(shape=(200, 200, 3)),
            Conv2D(32, (3, 3), activation='relu'),
            MaxPooling2D((2, 2)),
            Flatten(),
            Dense(128, activation='relu'),
            Dense(len(self.class_labels), activation='softmax')
        ])
        model.compile(optimizer=Adam(learning_rate=0.00001),
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])
        return model

    def update_progress(self, message):
        print(message)

    def update_genome(self, genome):
        print(f"Genome: {genome}")

    def update_full_iteration(self, full_iteration):
        print(f"Full Iteration: {full_iteration}")

    def train(self, epochs, max_steps_per_epoch=100):
        trainer = TrainingThread(self.model, self.dataset_path, self.target_size, epochs, self.class_labels, max_steps_per_epoch, GOOGLE_DRIVE_PATH)
        trainer.train_model()

# Example usage for training
app = CardPredictorApp()

# Train the model (example with 100 epochs)
app.train(epochs=10000)


Genome: 466
Full Iteration: 47871
Found 68436 images belonging to 20 classes.
Epoch 1/10000, Step 1/1000000: Loss=2.9820, Accuracy=0.0312
Progress: 0%
Epoch 1/10000, Step 2/1000000: Loss=2.9031, Accuracy=0.1406
Progress: 0%
Epoch 1/10000, Step 3/1000000: Loss=2.8958, Accuracy=0.1458
Progress: 0%
Epoch 1/10000, Step 4/1000000: Loss=2.8124, Accuracy=0.1719
Progress: 0%
Epoch 1/10000, Step 5/1000000: Loss=2.7317, Accuracy=0.2000
Progress: 0%
Epoch 1/10000, Step 6/1000000: Loss=2.6622, Accuracy=0.2135
Progress: 0%
Epoch 1/10000, Step 7/1000000: Loss=2.5893, Accuracy=0.2500
Progress: 0%
Epoch 1/10000, Step 8/1000000: Loss=2.5500, Accuracy=0.2578
Progress: 0%
Epoch 1/10000, Step 9/1000000: Loss=2.5085, Accuracy=0.2812
Progress: 0%
Epoch 1/10000, Step 10/1000000: Loss=2.4774, Accuracy=0.2844
Progress: 0%
Epoch 1/10000, Step 11/1000000: Loss=2.4360, Accuracy=0.2955
Progress: 0%
Epoch 1/10000, Step 12/1000000: Loss=2.4335, Accuracy=0.2995
Progress: 0%
Epoch 1/10000, Step 13/1000000: Loss=2.4397

KeyboardInterrupt: 

# Predict Image

In [ ]:
import os
import random
import numpy as np
from PIL import Image
from tensorflow.keras.models import load_model

# Path for Google Drive
GOOGLE_DRIVE_PATH = '/content/drive/My Drive/model/google_values.json'  # Adjust path as needed
MODELS_DIR = "/content/drive/My Drive/model"  # Set your desired models directory in Google Drive
DATASET_PATH = "/content/drive/My Drive/data"  # Dataset path remains the same

# Application for prediction (no GUI)
class CardPredictorApp:
    def __init__(self):
        self.dataset_path = DATASET_PATH
        self.target_size = (200, 200)
        self.class_labels = self.get_class_labels()
        self.model = self.load_or_create_model()

    def get_class_labels(self):
        labels = []
        for subdir in os.listdir(self.dataset_path):
            if os.path.isdir(os.path.join(self.dataset_path, subdir)):
                labels.append(subdir)
        return sorted(labels)

    def load_or_create_model(self):
        model_file = os.path.join(MODELS_DIR, 'card_predictor_model.keras')
        if os.path.exists(model_file):
            return load_model(model_file)
        else:
            print("Model not found. Please train the model first.")
            return None

    def start_prediction(self):
        # Get a random image from the dataset
        all_image_paths = []
        for subdir, _, files in os.walk(self.dataset_path):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    all_image_paths.append(os.path.join(subdir, file))

        if not all_image_paths:
            print("No images found in the dataset.")
            return

        random_image_path = random.choice(all_image_paths)
        print(f"Prediction for random image: {random_image_path}")

        img = Image.open(random_image_path).convert("RGB").resize(self.target_size, Image.BICUBIC)
        img_array = np.array(img) / 255.0
        img_array = np.expand_dims(img_array, axis=0)

        prediction = self.model.predict(img_array)
        predicted_class = np.argmax(prediction)

        print(f"Prediction: {self.class_labels[predicted_class]}")

# Example usage for prediction
app = CardPredictorApp()

# Test prediction with a random image from the dataset
app.start_prediction()
